# Processing Template
___
Jonathan Zhang

Here, we outline how to use this notebook to process stitched and background-subtracted images from HT-BAM experiments. Before you start, carefully read the guidelines below.

## General Notes

### Metadata framework
There exist two sources of metadata to bridge the gap between raw pixels and experimental variables:
- `imaging.csv`: This file logs hardware state and raster metadata for all images taken in an experiment. All information needed to stitch and background subtract images is found here. You may manually overwrite this file to alter how images are processed, but always be sure to create a backup copy beforehand.
    - For the sake of continuity, the stitching and background subtraction modules carry over this data into `stitched_images.csv` and `bgsub_images.csv`.
- `series_index.json`: Metadata specific to the experimental conditions (e.g., titration concentrations) is saved within individual series directories. This ensures that raw data remain linked to their biochemical context. This file is largely irrelevant during stitching, but become important at the processing stage.

### Before you begin
- Manually delete any images you do not want processed beforehand.
- Previous versions of the stitching code required manually directory reorginization. In this version, reorganizing the directory structure is entirely unnecessary and will likely cause the code to fail.
    - Do not rename, move, or create new files within directories of raw images or outputs from the stitching and background subtraction modules (i.e. `raw_images`, `stitched_images`, and `bgsub_images`).
- If you are running processing on a laptop, probably a good idea to close all uneeded applications (code is both memory and CPU intensive).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from htbam_analysis.stitching import ImageStitcher, BackgroundSubtractor
from htbam_analysis.processing import Processor
from htbam_analysis.processing.experiment import Experiment, Device, DataHandler

from pathlib import Path
import pandas as pd
import numpy as np

# to mute annoying numpy warnings
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Image Stitching
The stitching process is designed to be automated and data-driven:
- `stitch_images`: Once the root path is provided, this function acts as the primary orchestrator. It uses the raster parameters defined in `imaging.csv` to find and stitch all raw data associated with the experiment. There are three arguments:
    1. `root`: path to the folder containing experiment data.
    2. `acqui_ori`: specifies corner of the device where imaging starts. In our experiments, this is always from the top-right (i.e. `(True, False)`) 
    3. [OPTIONAL] `rotation`: global rotation parameter. This can drift over time and may need to be adjusted. If stitched images look jagged, this is the first thing that should be tuned. If not provided, rotation will be inferred automatically.
- `stitch_single_raster`: This is the lower-level function called by the main stitcher. If your file-naming convention or folder structure changes substantially from convention, you can bypass the automated discovery logic implemented in `stitch_images` and use this function directly by providing your own path-finding logic.

NOTE: by default, stitched images are not flat-field corrected. If one desires to implement correction for all or a subset of images, one can "hack" the ImageStitcher by manually updating the `apply_ff_correction` column in the `raster_data` attribute of the `ImageStitcher`. For example:

```python
# configure ff correction for just button quant images
bq_mask = stitcher.raster_data['image_path'].apply(str).str.contains('2026-03-09_15-16-09_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant')
stitcher.raster_data.loc[bq_mask, 'apply_ff_correction'] = True
```

In [ ]:
# init stitcher
root = Path('../../example_data/jacob/20260509')
stitcher = ImageStitcher(root)

# # configure ff correction for just button quant images
# bq_mask = stitcher.raster_data['image_path'].apply(str).str.contains('aura_cyan_2_5_sensitivity_2x2_1')
# stitcher.raster_data.loc[bq_mask, 'apply_ff_correction'] = True

# NOTE: SETUP 2 STOPGAP SOLUTION FOR BOOF STITCHES
# NOTE: ONLY RUN FOR SETUP 2 EXPERIMENTS UNTIL STAGE ISSUE IS FIXED
# stitcher.raster_data['raster_overlap'] = 0.105

Calculating rotation is now done automatically. Still want to test it yourself? Use the following function:

In [ ]:
# # OPTIONAL: test different rotation parameters to find the ideal one 
# stitcher.test_stitching_rotations(
#     raster_path = root/'raw_images/2026-05-09_16-58-08_d3_spectra_cyan_2_5_sensitivity_2x2_1_pre_button_quant_brightfield',
#     rotations = [2.2, 1.8, 1.6, 1.4, 1.2, 1.25, 1.0],
#     acqui_origin = (True, False),
#     outdir = root / 'stitch_test'
# )

In [ ]:
# stitch the images
stitcher.stitch_images( rotation=None,                  # Automatically find rotation.
                        acqui_origin=(True, False))

## Background Subtraction

Like stitching, background subtraction logic is split into two layers to balance automation with developer flexibility:
- `background_subtract_images`: The main function for background subtraction. It automatically groups images based on specified metadata headers (like channel or exposure_time) and applies the subtraction across the entire experiment. It uses `stitched_images.csv` as the source for metadata.
- `background_subtract`: The lower-level function, called within the above, which performs the background subtraction operation on a single image or array. If your file structure changes substantially or you need to implement custom subtraction logic, you can call this function directly and bypass the automated grouping.

### Note on Image Grouping
Users can specify metadata headers in `stitched_images.csv` to use for image grouping (i.e. exposure time, lightsource, etc.). If complex groupings are required, one can manually write new columns to `stitched_images.csv` to define custom logic (but always save a backup copy beforehand).

In [ ]:
# paths to all background images
# use full, absolute paths
# background_images = [
#     '/Volumes/JSZ2/20260309/stitched_images/2026-03-09_11-36-50_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant_background.tif',
#     '/Volumes/JSZ2/20260309/stitched_images/2026-03-09_15-19-34_d2_aura_green_3_50_sensitivity_2x2_1_mscarlet_background.tif',
#     '/Volumes/JSZ2/20260309/stitched_images/2026-03-09_17-23-13_d2_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif',
# ]
background_images = [ ]
settings_to_match = ['temp', 'hum','setup', 'dname', 'lightsource', 'channel', 'exposure', 'camera_mode', 'binning', 'nosepiece', 'apply_ff_correction']
subtractor = BackgroundSubtractor(root)
subtractor.subtract(
    background_images=background_images,
    settings_to_match=settings_to_match,
)

## Processing

### Initialization

To begin, initialize an `Experiment` object by providing the root directory of your data. You must then create `Device` objects, representing the devices used within an experiment, and then register them. 

When creating device objects, one can specify pinlists using the `set_pinlist` method. There are two kwargs that are helpful here:
    1. `block_descriptions`- a dictionary specifying which variants are present in each block (for flow-on experiments only)
    2. `pinlist_path`- path to a pinlist file output by `Array-Print` (for expression experiments only) 

For image and metadata management, we use the `DataHandler` class. This class is responsible for fetching images and associated metadata to be processed.

### Processing logic

The execution logic is as follows:
1. Retrieve image data and metadata to be processed via methods implemented in the `DataHandler`.
    - This is done by providing image identifier(s) to the `get_images()` method within the `DataHandler` class.
2. Initialize a processor object.
    - In addition to your experiment object and image data from (1), you will need to designate the type of feature to process (i.e. 'button', 'chamber', or 'all'). 
3. Optionally, set reference images for each device.
    - You'll need to designate the coordinates of the center of all four corner buttons (see below).
4. Process images using the core `process()` method.
    - If you don't have a reference image, pass a single set of coordinates OR a list of N coordinates (N being the number of images to be processed).

Poor feature finding results arise from two problems (almost all the time):
1. Jagged stitches. This can be fixed by going back to stitching and optimizing the rotation parameter.
2. Bad corner picks. Pick new corners.

### A note on corner picking

We have a command line tool for picking corners. Within your terminal, run the following command (be sure to activate the correct conda environment beforehand):

```bash
pick-corners /path/to/stitched/and/subtracted/image.tif
```

In [ ]:
# initialize experiment object
experiment = Experiment(root)

# initialize device(s)
# NOTE: dname NEEDS to exactly match that used within the experiment
d1 = Device(setup='s1', dname='d3', dims=(32, 56))
d1.set_pinlist(pinlist_path=root / 'your_array_pinlist.csv')

# d2 = Device(setup='s1', dname='d2', dims=(32, 56))
# d2.set_pinlist(block_descriptions={1: 'BLANK', 2: 'BLANK', 3: 'KRAS', 4: 'KRAS'})

# add devices to experiment
experiment.add_device(d1)
#experiment.add_device(d2)

# init data handler object
# will print out identifiers that can be passed into the get_images method
data_handler = DataHandler(root)

## Button Quant Processing

The `identifiers` argument can either be a list of identifiers (printed above by the `DataHandler` class), or a single identifier. All images associated with the identifier will be loaded.

In [ ]:
# copy/paste identifiers printed by DataHandler above
button_quant_identifiers = [
    '2026-05-10_13-02-25_d3_spectra_cyan_2_5_sensitivity_2x2_1_button_quant'
]

# get image data
button_quant_image_data = data_handler.get_images(identifiers=button_quant_identifiers)

# init the processor
button_quant_processor = Processor(
    experiment= experiment,
    image_data = button_quant_image_data,
    features = 'button'
)

# set corners for devices (use pick-corners app here)
button_quant_processor.set_corners('d3', [(417, 452), (6720, 452), (401, 6818), (6667, 6856)])

# process the data
button_quant_df = button_quant_processor.process()

# save the data to disc
button_quant_df.to_csv(root / 'button_quant.csv')

## Standard Curve Processing

In [ ]:
# copy/paste identifiers printed by DataHandler above
standard_images = data_handler.get_images(identifiers='2026-05-10_13-17-57_standard_curve_pbp')

standard_processor = Processor(experiment, image_data=standard_images, features='chamber')

# set corners for devices (use pick-corners app here)
standard_processor.set_corners('d3', [(416, 451), (6720, 461), (401, 6818), (6673, 6862)])

# set references (for both devices)
standard_processor.set_reference(
    image=root / 'bgsub_images/2026-05-10_13-17-57_standard_curve_pbp/2026-05-10_13-58-08_d3_spectra_blue_6_5_sensitivity_2x2_1_standard_curve_pbp_7.tif',
    coerce_chamber_center=False
    )

standard_df = standard_processor.process(use_reference=True)

In [ ]:
# add your desired output path here
standard_df.to_csv(root / 'standard_data.csv.bz2', compression='bz2')

## Kinetics Processing

In [ ]:
# copy/paste identifiers printed by DataHandler above
kinetics_images = data_handler.get_images(identifiers='2026-05-10_00-13-33_kinetic_series_MM_Kinetics')

kinetics_processor = Processor(experiment, image_data=kinetics_images, features='chamber')

# corners for each image (use pick-corners app here)
kinetics_processor.set_corners('d3', [(224, 707), (6525, 528), (395, 7073), (6664, 6927)])

# set references (for both devices)
kinetics_processor.set_reference( 
   image=root / 'bgsub_images/2026-05-10_00-13-33_kinetic_series_MM_Kinetics/2026-05-10_00-13-33_timecourse_0/2026-05-10_00-18-43_d3_spectra_blue_6_5_sensitivity_2x2_1_kinetic_series_MM_Kinetics_timecourse_0_0.tif',
   coerce_chamber_center=False
   )

In [ ]:
kinetics_df = kinetics_processor.process(use_reference=False,)

Have multiple sets of kinetics images? Repeat the above steps for each additional folder of images, to get another output dataframe (like kinetics_df). Then, concatenate the dataframes together with:

```kinetics_df_merged = pd.concat([kinetics_df_1, kinetics_df_2], ignore_index=True)```

Finally, output the CSV:

In [ ]:
# add your desired output path here
kinetics_df.to_csv(root / 'kinetics_data.csv.bz2', compression='bz2')